In [1]:
# 03_model_training.ipynb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import os,sys

project_root = os.path.abspath("..")   # ← 根據你的資料夾結構調整
sys.path.insert(0, project_root)

print("Current working dir:", os.getcwd())
print("Project root added:", project_root)

Current working dir: /Users/mankin/Desktop/Anomaly_Detection_on_Graph/notebooks
Project root added: /Users/mankin/Desktop/Anomaly_Detection_on_Graph


In [2]:
# 確保每次訓練前都從頭開始處理資料，避免舊的 processed 檔案影響結果
processed_path = "../data/processed/elliptic_processed.pt"
if os.path.exists(processed_path):
    os.remove(processed_path)
    print("已成功刪除舊的 processed 檔案！下次執行 EllipticDataset 會重新處理")
else:
    print("找不到舊檔案，已經是乾淨狀態")

已成功刪除舊的 processed 檔案！下次執行 EllipticDataset 會重新處理


In [3]:
# ====================== 設定 Random Seed（確保實驗可重現） ======================

from src.utils import set_seed
set_seed(42)

 Random seed 已固定為 42，所有隨機性已關閉，實驗結果可完全重現。


In [4]:
# ====================== 引入 config ======================

from src.config import Config
cfg = Config()               

In [ ]:
#====================== override 你想改的參數 ======================
    
cfg.exp_name = "exp001_sage_pipeline_concat"        # ！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
cfg.model_name = "GraphSAGE"              # default 是 GraphSAGE，如果你想試試看 GAT 就改成 "GAT"
cfg.use_pipeline = True          # Pipeline or end-to-end training

cfg.hidden_dim = 32
cfg.num_layers = 3
cfg.lr = 0.002              
cfg.epochs = 10                  
cfg.patience = 25                  
cfg.dropout = 0.45

cfg.use_degree = False
cfg.use_pagerank = False
cfg.use_clustering = False  
cfg.use_eigenvector = False    # False 
cfg.use_betweenness = False         # very long to compute, so default is False

#cfg.aggregator = "mean"            # 如果是 GraphSAGE 才需要，改成 "mean"、"pool"、"lstm"、"gcn"
#cfg.heads = 8                      # 如果是 GAT 才需要

In [6]:
# ====================== Create Experiment Folder ======================

from src.utils import create_experiment

# Create experiment directory and save config.yaml
exp_dir = create_experiment(
    cfg=cfg,
    description=""   # 實驗描述（可選）e.g. GraphSAGE with degree + PageRank features, temporal split, mean aggregator
)

print(f"實驗目錄建立完成：{exp_dir}")

本次實驗名稱：exp_test2
實驗資料夾：../experiments/exp_test2
config.yaml 已自動建立
實驗設定已儲存至: ../experiments/exp_test2/config.yaml
實驗目錄建立完成：../experiments/exp_test2


In [7]:
# ====================== 載入資料 ======================

# start before initialize dataset, to ensure processed file is deleted
from src.data import EllipticDataset   # 從 src.data 模組中引入 EllipticDataset 類別，這是我們之前定義的資料集類別，用來處理 Elliptic 資料集。

dataset = EllipticDataset(
    root=str(cfg.data_dir), 
    use_degree=cfg.use_degree,
    use_pagerank=cfg.use_pagerank,
    use_clustering=cfg.use_clustering,
    use_eigenvector=cfg.use_eigenvector,
    use_betweenness=cfg.use_betweenness
)
data = dataset[0]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

x = data.x.to(device)
edge_index = data.edge_index.to(device)
y = data.y.to(device)

print(f"資料載入完成：{x.shape[0]} nodes, {edge_index.shape[1]} edges, {x.shape[1]} features")

/opt/anaconda3/envs/GAD/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing...


開始處理 Elliptic 資料集...
MARKER: process() method called - About to read classes
Nodes: 203769, Features dim: 165
Sample txIds (col 0): [230425980, 5530458, 232022460, 232438397, 230460314]
Sample time steps (col 1): [1, 1, 1, 1, 1]
edgelist 有效行數: 234355
Sample edges txId1 (first 5): [230425980, 232022460, 230460314, 230333930, 232013274]
MARKER: Starting edge matching loop
有效 directed edges: 234355
匹配成功率: 100.00% (234355/234355)
無向 edges: 468710
加入 degree → x dim: 166


/Users/mankin/Desktop/Anomaly_Detection_on_Graph/src/data.py:225: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  torch.save((self.data, self.slices), self.processed_paths[0])


處理完成！
最終: 203769 nodes, 468710 edges
資料載入完成：203769 nodes, 468710 edges, 166 features


Done!
/opt/anaconda3/envs/GAD/lib/python3.11/site-packages/torch_geometric/data/in_memory_dataset.py:131: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([DataEdgeAttr])` to allowlist this global.
  out = fs.torch_load(path)


In [8]:
# ====================== Initialize Model ======================

if cfg.model_name.upper() == "GRAPHSAGE":           # GraphSAGE
    from src.models import ImprovedGraphSAGE
    model = ImprovedGraphSAGE(
        in_channels=x.shape[1],
        hidden_dim=cfg.hidden_dim,        
        num_layers=cfg.num_layers,        
        dropout=cfg.dropout,  
        aggregator=cfg.aggregator  
    ).to(device)

elif cfg.model_name.upper() == "GAT":               # GAT 
    from src.models import ImprovedGAT
    model = ImprovedGAT(
        in_channels=x.shape[1],          
        hidden_dim=cfg.hidden_dim,
        num_layers=cfg.num_layers,
        heads=getattr(cfg, "heads", 8),            # ← 使用 getattr 更安全
        dropout=cfg.dropout
    ).to(device)

else:
    raise ValueError(f"不支援的模型名稱: {cfg.model_name}")

print(model)

 Kaiming initialization + Residual Connections applied (layers=3, hidden=32)
ImprovedGraphSAGE(in=166, hidden=32, layers=3, agg=mean, dropout=0.45, init=Kaiming+Residual+Pipeline)


In [9]:
# ====================== Data Splitting ======================

from src.split import split_data

# Perform data split (temporal split is recommended)
train_idx, val_idx, test_idx = split_data(
    data=data,
    y=y,
    device=device,
    temporal_split=True,          # Set to False if you want random split
    test_time_threshold=34,
    val_size=0.2,
    random_state=42
)

使用 Temporal Split（推薦）
訓練: 23915 | 驗證: 5979 | 測試: 16670


In [10]:
# ====================== Training + Evaluation + Saving ======================

from src.train import train   # ← 只 import 這一個函數

best_val_auc, best_model_path, test_auc, test_auprc, best_epoch = train(
    model=model, 
    x=x, 
    edge_index=edge_index, 
    y=y,
    train_idx=train_idx, 
    val_idx=val_idx, 
    test_idx=test_idx,
    cfg=cfg,
    device=device, 
    exp_dir=exp_dir
)

print(f"\n Training finished!")
print(f"Test AUC: {test_auc:.4f} | AUPRC: {test_auprc:.4f}")

 使用 End-to-End 模式 (GraphSAGE + CrossEntropy)
 Using Weighted CrossEntropy Loss with weights: [1.0, 15.0]
 Training model: GraphSAGE | hidden_dim=32, layers=3, dropout=0.45
 Training started at 2026-03-31 01:32:50

Epoch   0 | Loss: 1.1135 | Val AUC: 0.7519 | Patience: 0/25
Epoch   5 | Loss: 0.8412 | Val AUC: 0.8610 | Patience: 0/25
Epoch   9 | Loss: 0.6769 | Val AUC: 0.8878 | Patience: 0/25

 Training finished in 4.8 seconds (0.08 minutes)
   Best model at epoch 9 (Val AUC = 0.8878)
   Early stopping patience used after best: 0
Best model saved → ../saved_models/graphsage_best_20260331_013254.pt
results.json 已儲存至: ../experiments/exp_test2/results.json (包含訓練時間紀錄)
已複製最佳模型 → ../experiments/exp_test2/model_best.pt

實驗 exp_test2 所有結果已儲存至：
   ../experiments/exp_test2/
   ├── config.yaml
   ├── results.json
   └── model_best.pt

 實驗完成！
   名稱：exp_test2
   路徑：../experiments/exp_test2
   檔案：
     • config.yaml
     • results.json
     • model_best.pt

 Training finished!
Test AUC: 0.7453 | AUPRC